In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from scipy import signal as sp_signal
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from joblib import Parallel, delayed  # <--- FOR PARALLEL PROCESSING
from google.colab import drive

# ==========================================
# 0. SETUP & MOUNT DRIVE
# ==========================================
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_ROOT = "/content/drive/MyDrive/capture24"
PROCESSED_DIR = os.path.join(DATA_ROOT, "processed_nb")
CHECKPOINT_DIR = os.path.join(DATA_ROOT, "checkpoints") # <--- NEW CHECKPOINT FOLDER
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Constants
FS = 100.0
WINDOW_SEC = 4.0
GRAVITY_WIN_SEC = 10
LOWPASS_CUTOFF = 20.0
PATCH_SIZE = 20
PATCH_DIM = 60
D_MODEL = 256
N_HEAD = 8
NUM_LAYERS = 6
DROPOUT = 0.1
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ Config Loaded. Device: {DEVICE}")
print(f"📂 Checkpoints will be saved to: {CHECKPOINT_DIR}")

# ==========================================
# 2. ROBUST PARALLEL DATA PROCESSING ENGINE
# ==========================================
# ==========================================
# REPLACEMENT: ROBUST COLUMN FINDER
# ==========================================
def process_single_file(path):
    """Worker function to process ONE file (FIXED COLUMN DETECTION)."""
    try:
        out_name = os.path.basename(path).replace(".csv.gz", "_processed.npz")
        out_path = os.path.join(PROCESSED_DIR, out_name)

        # Skip if already exists
        if os.path.exists(out_path):
            return "SKIPPED"

        # Load dataframe
        df = pd.read_csv(path, compression="gzip")

        # --- FIXED COLUMN DETECTION ---
        # 1. Find Time: Look for 'time', 'timestamp', 'ts', 'datetime'
        t_col = next((c for c in df.columns if any(x in c.lower() for x in ['time', 'ts', 'date'])), None)

        # 2. Find Accel: Look for 'acc' OR just 'x', 'y', 'z'
        # Strategy A: Look for explicit 'acc' prefix
        a_cols = [c for c in df.columns if 'acc' in c.lower()][:3]

        # Strategy B: If A fails, look for exact 'x', 'y', 'z'
        if len(a_cols) < 3:
            a_cols = [c for c in df.columns if c.lower() in ['x', 'y', 'z']][:3]

        # Strategy C: If B fails, try columns 1, 2, 3 (assuming col 0 is time)
        if len(a_cols) < 3 and len(df.columns) >= 4:
            # Heuristic: assume time is 0, acc is 1,2,3
            a_cols = df.columns[1:4]

        if t_col is None or len(a_cols) < 3:
            return f"❌ Skipped {os.path.basename(path)}: Columns missing. Found: {list(df.columns)}"
        # -------------------------------

        t = pd.to_datetime(df[t_col], utc=True).astype('int64').values / 1e9
        X = df[a_cols].values.astype(float)

        # Resample
        t0, t1 = t[0], t[-1]
        n_samples = int((t1 - t0) * FS) + 1
        t_u = np.linspace(t0, t1, n_samples)
        Xu = np.zeros((n_samples, 3))
        for i in range(3):
            f = interp1d(t, X[:, i], bounds_error=False, fill_value="extrapolate")
            Xu[:, i] = f(t_u)

        # Filter
        Xc = Xu.copy()
        win = int(GRAVITY_WIN_SEC * FS)
        for i in range(3):
            roll = pd.Series(Xu[:, i]).rolling(window=win, center=True, min_periods=1).mean().values
            Xc[:, i] = Xu[:, i] - roll

        nyq = FS / 2
        b, a = sp_signal.butter(4, LOWPASS_CUTOFF / nyq, btype='low')
        Xf = sp_signal.filtfilt(b, a, Xc, axis=0)

        # Save
        np.savez_compressed(out_path, accel=Xf, fs=FS)
        return "SUCCESS"

    except Exception as e:
        return f"❌ Error {os.path.basename(path)}: {str(e)}"

def run_parallel_processing():
    raw_files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.csv.gz")))
    print(f"Found {len(raw_files)} raw files.")

    LIMIT = None  # Process ALL files
    files_to_process = raw_files[:LIMIT] if LIMIT else raw_files

    print(f"🚀 Starting Robust Processing on {len(files_to_process)} files...")
    # Reduced n_jobs to 4 to prevent memory crashes
    results = Parallel(n_jobs=4, verbose=5)(
        delayed(process_single_file)(f) for f in files_to_process
    )

    # Analysis
    success = results.count("SUCCESS")
    skipped = results.count("SKIPPED")
    errors = [r for r in results if "❌" in r]

    print(f"\n🎉 Processing Summary:")
    print(f"   ✅ Success: {success}")
    print(f"   ⏭️ Skipped (Already Done): {skipped}")
    print(f"   ❌ Errors: {len(errors)}")

    if errors:
        print("   Sample Errors:", errors[:5])

# REPLACE Section 2 in your code with this block and Run.

🔄 Mounting Google Drive...
Mounted at /content/drive
✅ Config Loaded. Device: cuda
📂 Checkpoints will be saved to: /content/drive/MyDrive/capture24/checkpoints


In [ ]:
import shutil

# 1. Delete old checkpoints so training starts fresh
checkpoint_dir = "/content/drive/MyDrive/capture24/checkpoints"
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    os.makedirs(checkpoint_dir)
    print("🗑️ Old checkpoints deleted. Ready for fresh 'Big Thing' training.")
else:
    print("✅ No old checkpoints found. Ready to go.")

🗑️ Old checkpoints deleted. Ready for fresh 'Big Thing' training.


In [ ]:
# ==========================================
# 6. MAIN EXECUTION (Restarted for Full Data)
# ==========================================
if __name__ == "__main__":
    # A. Load Data (Already processed, so this will be fast)
    try:
        # We assume patches are already loaded in memory from your last step
        # If 'patches' variable is lost, uncomment the next line:
        # patches = load_all_patches()

        print(f"🚀 Starting Training on {len(patches)} samples...")

        full_ds = IMUMaskedDataset(patches, mode="ssl")
        train_size = int(0.8 * len(full_ds))
        train_ds, val_ds = random_split(full_ds, [train_size, len(full_ds) - train_size])

        # Increase batch size for A100 to go faster
        BIG_BATCH_SIZE = 512
        train_loader = DataLoader(train_ds, batch_size=BIG_BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BIG_BATCH_SIZE)

        # C. Train SSL (Fresh Start)
        print("\n--- Stage 1: SSL Pre-training (Full Data) ---")
        ssl_model = ImprovedMaskedTransformer().to(DEVICE)
        # Train for 3 epochs (sufficient for 3.5M samples)
        ssl_model = train_with_checkpoints(ssl_model, train_loader, "ssl_pretrain", epochs=3)

        # D. Train Forecasting
        print("\n--- Stage 2: Forecasting (Full Data) ---")
        train_ds.dataset.mode = "forecast"
        forecast_model = ImprovedMaskedTransformer().to(DEVICE)
        forecast_model.load_state_dict(torch.load("best_ssl_pretrain.pth"))
        forecast_model = train_with_checkpoints(forecast_model, train_loader, "forecast_finetune", epochs=3)

        print("\n✅ BIG THING COMPLETE! You have trained on the full dataset.")

    except Exception as e:
        print(f"\n❌ Pipeline Stopped: {e}")


❌ Pipeline Stopped: name 'patches' is not defined


In [ ]:
# ==========================================
# 7. FINAL STAGE: HAR ON FULL DATASET (FIXED)
# ==========================================

# 1. Define the Missing Classifier Class
class ClassifierHead(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        # Simple head: 256 -> 128 -> 2 (Active/Sedentary)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        # 1. Embed & Encode
        emb = self.base.ln(self.base.input_emb(x))
        feat = self.base.encoder(emb)

        # 2. Global Pooling (Mean across 20 patches)
        pool = feat.mean(dim=1)

        # 3. Classify
        return self.head(pool)

# 2. Run the Efficiency Experiment
print("\n--- 🚀 Stage 3: HAR Label Efficiency (Full Scale) ---")

# Setup Dataset
# If 'patches' is lost from memory, uncomment:
# patches = load_all_patches()

full_ds = IMUMaskedDataset(patches, mode="classification")
train_size = int(0.8 * len(full_ds))
train_ds, val_ds = random_split(full_ds, [train_size, len(full_ds) - train_size])
val_loader = DataLoader(val_ds, batch_size=512)

# Test different data fractions
fractions = [0.001, 0.01, 0.1, 1.0]
accuracies = []

for frac in fractions:
    # Subset the data
    subset_len = int(frac * len(train_ds))
    if subset_len < 10: subset_len = 10

    indices = torch.randperm(len(train_ds))[:subset_len]
    sub_loader = DataLoader(Subset(train_ds, indices), batch_size=512, shuffle=True)

    # Init Classifier with BEST SSL Weights
    base = ImprovedMaskedTransformer().to(DEVICE)
    base.load_state_dict(torch.load("best_ssl_pretrain.pth"))
    clf = ClassifierHead(base).to(DEVICE)

    optimizer = optim.Adam(clf.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    print(f"Training on {frac*100}% Data ({subset_len} samples)...")

    # Quick Train (1 epoch is enough for this demo)
    clf.train()
    for x, _, y in sub_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = clf(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    # Evaluate
    correct = 0
    total = 0
    clf.eval()
    with torch.no_grad():
        for x, _, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = clf(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct/total * 100
    accuracies.append(acc)
    print(f"   -> Accuracy: {acc:.2f}%")

# 3. Plot the Final Graph
plt.figure(figsize=(8,5))
plt.plot([f*100 for f in fractions], accuracies, 'bo-', linewidth=2)
plt.title(f"Slide 16: Label Efficiency on {len(full_ds)//1000}k Samples")
plt.xlabel("% Training Data")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 105)
plt.grid(True)
for x, y in zip(fractions, accuracies):
    plt.text(x*100, y+2, f"{y:.1f}%", ha='center')
plt.show()

print("\n✅ BIG THING COMPLETE. Save this graph for Slide 16!")


--- 🚀 Stage 3: HAR Label Efficiency (Full Scale) ---


NameError: name 'IMUMaskedDataset' is not defined

In [ ]:
# ==========================================
# 8. FORECASTING HORIZON ANALYSIS (Slide 14/15)
# ==========================================
print("\n--- 🚀 Stage 4: Forecasting Horizon Analysis ---")

# 1. Setup Data for Forecasting Evaluation
# We need a test set in 'forecast' mode
# (Assuming 'patches' and 'val_ds' are available from previous steps)
val_ds.dataset.mode = "forecast"
val_loader = DataLoader(val_ds, batch_size=512)

# 2. Load the Best Forecasting Model
model = ImprovedMaskedTransformer().to(DEVICE)
model.load_state_dict(torch.load("best_forecast_finetune.pth"))
model.eval()

# 3. Calculate Error per Time Step
# Forecast length is 2 patches (40 samples @ 100Hz = 0.4s)
# Or if you changed it to match the slide (0.5s - 2.0s), adjust PRED_LEN
PRED_LEN = 40
errors = np.zeros(PRED_LEN)
counts = 0

print("Calculating error per time step...")
with torch.no_grad():
    for x, mask, _ in val_loader:
        x, mask = x.to(DEVICE), mask.to(DEVICE)

        # Predict
        pred = model(x)

        # Get only the forecasted part (masked part)
        # x shape: [Batch, 20_patches, 60_dim] -> Flatten to time
        # We need to reshape to [Batch, Time_Steps, Axes] to match horizon
        # 20 patches * 20 samples = 400 samples

        x_time = x.view(x.size(0), -1, 3)      # [Batch, 400, 3]
        pred_time = pred.view(pred.size(0), -1, 3)

        # The mask was on the last 2 patches = last 40 samples
        # So we look at the last 40 steps
        horizon_true = x_time[:, -PRED_LEN:, :]
        horizon_pred = pred_time[:, -PRED_LEN:, :]

        # MSE per time step (average over batch and 3 axes)
        # Result shape: [Batch, PRED_LEN, 3] -> Mean over Batch & Axes -> [PRED_LEN]
        diff = (horizon_true - horizon_pred) ** 2
        mse_step = torch.mean(diff, dim=(0, 2))

        errors += mse_step.cpu().numpy() * x.size(0) # Weighted sum
        counts += x.size(0)

# Average error
final_errors = errors / counts

# 4. Plot the Curve
time_axis = np.arange(1, PRED_LEN + 1) / FS # Time in seconds

plt.figure(figsize=(8, 5))
plt.plot(time_axis, final_errors, 'r-', linewidth=3, label="SSL-Pretrained")
# Optional: Plot a flat baseline or comparison if you have it
# plt.plot(time_axis, [0.0004]*len(time_axis), 'k--', label="Baseline (Approx)")

plt.title("Forecasting Error vs. Horizon (Full Data)")
plt.xlabel("Forecast Horizon (seconds)")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("✅ Horizon Graph Generated.")